# Notebook 10: Alpha Appendix

This is a **compact reviewer-facing appendix**.
Rows are filtered by a fixed **alpha completeness gate** before display, so this is **complete rows only**.
Missing or incomplete runs are intentionally omitted.

This notebook is **not a deep validation notebook**; it is an accounting-style view of what is complete and what is not.


In [1]:
import sys
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    from notebooks._helpers import (
        repo_root,
        load_metrics_master,
        load_run_manifest,
        ensure_run_status_norm,
        ensure_optional_columns,
        ensure_numeric_columns,
        parse_metric_availability,
        unavailable_panel_table,
    )
except ModuleNotFoundError:
    from _helpers import (
        repo_root,
        load_metrics_master,
        load_run_manifest,
        ensure_run_status_norm,
        ensure_optional_columns,
        ensure_numeric_columns,
        parse_metric_availability,
        unavailable_panel_table,
    )
ROOT = repo_root()

# Ensure src module is importable
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.dqi_pipeline import has_complete_alpha_metrics
metrics_master = load_metrics_master(ROOT / "results", required=False)

focus = metrics_master.copy()
if not focus.empty:
    focus = ensure_run_status_norm(focus)

if focus.empty:
    alpha_scope = pd.DataFrame(columns=focus.columns)
    stage_filter = "metrics_master unavailable"
else:
    alpha_scope = focus.copy()
    # Primary filter: look for stage=alpha_validation
    if "stage" in alpha_scope.columns:
        alpha_validation_rows = alpha_scope[alpha_scope["stage"].astype(str).str.lower() == "alpha_validation"].copy()
        if not alpha_validation_rows.empty:
            alpha_scope = alpha_validation_rows
            stage_filter = "stage == alpha_validation"
        else:
            # Fallback: Use Family C rows with coherent execution mode from matrix_b
            # These are the alpha/coherent validation runs stored under benchmark_matrix stage
            family_c_rows = alpha_scope[
                (alpha_scope["family"].astype(str).eq("C")) &
                (alpha_scope["matrix"].astype(str).eq("matrix_b"))
            ].copy() if "family" in alpha_scope.columns and "matrix" in alpha_scope.columns else pd.DataFrame()
            
            if not family_c_rows.empty:
                alpha_scope = family_c_rows
                stage_filter = "fallback: family=C + matrix=matrix_b (alpha validation data)"
            else:
                alpha_scope = pd.DataFrame(columns=alpha_scope.columns)
                stage_filter = "no alpha_validation or family_c matrix_b rows"
    else:
        stage_filter = "no stage column"

# Status field normalization with defensive fallback handling
# Different data sources may use different column names for run status:
# - "status" (legacy format)
# - "run_status" (current benchmark format)
# - "run_status_norm" (normalized format from ensure_run_status_norm)
# This defensive code ensures the appendix table always has a "status" column
# regardless of which source format was used.
if not alpha_scope.empty and "status" not in alpha_scope.columns:
    if "run_status" in alpha_scope.columns:
        alpha_scope["status"] = alpha_scope["run_status"]
    elif "run_status_norm" in alpha_scope.columns:
        alpha_scope["status"] = alpha_scope["run_status_norm"]
    else:
        alpha_scope["status"] = "unavailable"

if not alpha_scope.empty:
    alpha_scope["status"] = alpha_scope["status"].where(alpha_scope["status"].notna(), "unavailable")
    alpha_scope.loc[alpha_scope["status"].astype(str).str.strip() == "", "status"] = "unavailable"

def _first_existing_col(frame: pd.DataFrame, candidates: list[str]) -> str | None:
    for candidate in candidates:
        if candidate in frame.columns and frame[candidate].notna().any():
            return candidate
    return None

# Success probability fallback chain: success_prob -> postselection_success -> decoder_success_rate
# This handles different metric naming conventions across data sources
success_probability = pd.Series(index=alpha_scope.index, dtype=float)
if not alpha_scope.empty:
    for idx, row in alpha_scope.iterrows():
        value = row.get("success_prob")
        if pd.isna(value):
            value = row.get("postselection_success")
        if pd.isna(value):
            value = row.get("decoder_success_rate")
        success_probability.loc[idx] = value

alpha_scope = alpha_scope.copy()
alpha_scope["success_probability"] = success_probability

# Add best_F fallback column using best_sampled_F
if not alpha_scope.empty:
    best_f_col = alpha_scope.get("best_F")
    best_sampled_f_col = alpha_scope.get("best_sampled_F")
    if best_f_col is not None and best_sampled_f_col is not None:
        alpha_scope["best_F"] = best_f_col.where(best_f_col.notna(), best_sampled_f_col)
    elif best_f_col is None and best_sampled_f_col is not None:
        alpha_scope["best_F"] = best_sampled_f_col

completeness_gate_applied = alpha_scope.apply(has_complete_alpha_metrics, axis=1) if not alpha_scope.empty else pd.Series(dtype=bool)
appendix_candidates = alpha_scope[completeness_gate_applied].copy() if not alpha_scope.empty else pd.DataFrame(columns=alpha_scope.columns)

artifact_col = _first_existing_col(
    alpha_scope,
    ["run_id", "artifact", "artifact_id", "run_key", "instance_id", "instance_name", "run_name", "artifact_name"],
)
decoder_col = _first_existing_col(alpha_scope, ["decoder", "decoder_name", "decoder_norm"])

if not appendix_candidates.empty:
    if artifact_col:
        artifact_or_run_id = appendix_candidates[artifact_col].where(appendix_candidates[artifact_col].notna(), "unavailable").astype(str)
    else:
        artifact_or_run_id = pd.Series([f"row_{idx}" for idx in appendix_candidates.index], index=appendix_candidates.index)
    if decoder_col:
        decoder = appendix_candidates[decoder_col].where(appendix_candidates[decoder_col].notna(), "n/a")
    else:
        decoder = pd.Series("n/a", index=appendix_candidates.index)
    dist_metric = appendix_candidates.get("coherent_vs_mixture_distribution_distance")
    if dist_metric is None:
        dist_metric = pd.Series(index=appendix_candidates.index, dtype=float)

    appendix_table = pd.DataFrame(
        {
            "artifact_or_run_id": artifact_or_run_id,
            "alpha_mode": appendix_candidates.get("alpha_mode"),
            "execution_mode": appendix_candidates.get("execution_mode"),
            "decoder": decoder,
            "best_F": appendix_candidates.get("best_F"),
            "success_probability": appendix_candidates.get("success_probability"),
            "top1_regret": appendix_candidates.get("top1_regret"),
            "distribution_metric": dist_metric,
            "status": appendix_candidates.get("status"),
        }
    )
    appendix_table = appendix_table[["artifact_or_run_id", "alpha_mode", "execution_mode", "decoder", "best_F", "success_probability", "top1_regret", "distribution_metric", "status"]]
    appendix_table["best_F"] = pd.to_numeric(appendix_table["best_F"], errors="coerce")
    appendix_table["success_probability"] = pd.to_numeric(appendix_table["success_probability"], errors="coerce")
    appendix_table["top1_regret"] = pd.to_numeric(appendix_table["top1_regret"], errors="coerce")
    appendix_table["distribution_metric"] = pd.to_numeric(appendix_table["distribution_metric"], errors="coerce")
    appendix_table = appendix_table.sort_values(["artifact_or_run_id", "alpha_mode", "execution_mode", "decoder", "status"], kind="mergesort").reset_index(drop=True)
    appendix_table_trimmed = appendix_table.head(30).copy()
else:
    appendix_table = pd.DataFrame(columns=["artifact_or_run_id", "alpha_mode", "execution_mode", "decoder", "best_F", "success_probability", "top1_regret", "distribution_metric", "status"])
    appendix_table_trimmed = appendix_table.copy()

In [2]:
filter_mode = "primary_stage_family"
run_manifest = load_run_manifest(ROOT / "results", required=False)
if "alpha_scope" not in globals():
    baseline_metrics = load_metrics_master(ROOT / "results", required=False)
    if not baseline_metrics.empty:
        baseline_metrics = ensure_run_status_norm(baseline_metrics)
        baseline_scope = baseline_metrics.copy()
        # Primary filter: look for stage=alpha_validation
        if "stage" in baseline_scope.columns:
            alpha_validation_rows = baseline_scope[baseline_scope["stage"].astype(str).str.lower() == "alpha_validation"].copy()
            if not alpha_validation_rows.empty:
                baseline_scope = alpha_validation_rows
            else:
                # Fallback: Use Family C rows from matrix_b (alpha validation data)
                if "family" in baseline_scope.columns and "matrix" in baseline_scope.columns:
                    family_c_rows = baseline_scope[
                        (baseline_scope["family"].astype(str).eq("C")) &
                        (baseline_scope["matrix"].astype(str).eq("matrix_b"))
                    ].copy()
                    if not family_c_rows.empty:
                        baseline_scope = family_c_rows
                    else:
                        baseline_scope = pd.DataFrame(columns=baseline_scope.columns)
                else:
                    baseline_scope = pd.DataFrame(columns=baseline_scope.columns)
    else:
        baseline_scope = pd.DataFrame(columns=baseline_metrics.columns)
else:
    baseline_scope = alpha_scope.copy()
if not baseline_scope.empty:
    baseline_scope = ensure_run_status_norm(baseline_scope)
focus = baseline_scope.copy()
if not focus.empty and "run_status_norm" in focus.columns:
    focus = focus[focus["run_status_norm"] == "completed"].copy()
ok_rows = focus.copy()
for col in [
    "coherent_mode_record",
    "coherent_vs_mixture_delta_best_F",
    "coherent_vs_mixture_delta_best_top_G_sampled_F",
    "coherent_vs_mixture_delta_postselection_success",
    "coherent_vs_mixture_distribution_distance",
]:
    if col not in ok_rows.columns:
        ok_rows[col] = np.nan
    if col not in focus.columns:
        focus[col] = np.nan
if focus.empty:
    display(
        unavailable_panel_table(
            panel="Alpha finite-size focus",
            reason="alpha_validation_unavailable",
            details="No completed alpha_validation or Family C matrix_b rows are available in metrics_master.",
        )
    )

In [3]:
required_baseline_fields = [
    "counterpart_key",
    "instance_id",
    "encoding",
    "decoder",
    "alpha_mode",
    "execution_mode",
    "canonical_trial_seed",
]
baseline_metadata_complete = (not baseline_scope.empty) and all(field in baseline_scope.columns for field in required_baseline_fields)
if not baseline_metadata_complete:
    display(
        unavailable_panel_table(
            panel="Fixed-baseline metadata",
            reason="baseline_metadata_unavailable",
            details="Missing one or more required baseline metadata fields.",
        )
    )


## Coverage and compact appendix panel

This section reports what was considered, what was kept by the completeness gate, and what survives for reviewer display.


In [4]:
total_candidates = len(alpha_scope)
retained_rows = len(appendix_candidates)
excluded_rows = total_candidates - retained_rows
completeness_required_fields = [
    "alpha_mode",
    "execution_mode",
    "status",
    "best_F",
    "success_probability",
    "(from success_prob, with fallback to postselection_success, then decoder_success_rate)",
    "top1_regret",
]

coverage_panel = pd.DataFrame(
    {
        "metric": [
            "total_candidate_rows_examined",
            "rows_retained_after_alpha_completeness_filter",
            "rows_excluded_by_completeness_filter",
            "scope_filter",
            "required_fields_for_completeness",
        ],
        "value": [
            total_candidates,
            retained_rows,
            excluded_rows,
            stage_filter,
            ", ".join(completeness_required_fields),
        ],
    }
)

display(Markdown("### Coverage / audit"))
display(coverage_panel)

if retained_rows == 0:
    display(Markdown("No complete alpha/coherent rows are available after the filter; this appendix is therefore reporting completeness accounting rather than comparative evidence at this point."))
else:
    display(Markdown(f"### Compact reviewer table (trimmed to first {len(appendix_table_trimmed)} of {retained_rows} complete rows)"))
    display(appendix_table_trimmed)

if retained_rows > 0:
    execution_modes = {str(v).strip().lower() for v in appendix_table.get("execution_mode", pd.Series(dtype=object)).dropna()}
else:
    execution_modes = set()
has_coherent = "coherent" in execution_modes
has_mixture = "mixture" in execution_modes
meaningful_alpha_coherent_view = retained_rows >= 2 and has_coherent and has_mixture

display(Markdown("### Interpretation"))
if meaningful_alpha_coherent_view:
    display("- The appendix currently has complete rows in both execution modes; a limited reviewer-facing alpha/coherent comparison remains possible for the shown slice.")
else:
    reasons = []
    if retained_rows < 2:
        reasons.append("too few complete rows")
    if not has_coherent:
        reasons.append("no complete coherent rows")
    if not has_mixture:
        reasons.append("no complete mixture rows")
    reason_text = ", ".join(reasons) if reasons else 'limited row diversity'
    display(
        Markdown("- The appendix does **not** currently support a meaningful alpha/coherent comparison at this slice (" + reason_text + " ); it should be read primarily as a completeness/accounting appendix.")
    )


### Coverage / audit

,metric,value
0,total_candidate_rows_examined,48
1,rows_retained_after_alpha_completeness_filter,42
2,rows_excluded_by_completeness_filter,6
3,scope_filter,fallback: family=C + matrix=matrix_b (alpha va...
4,required_fields_for_completeness,"alpha_mode, execution_mode, status, best_F, su..."


### Compact reviewer table (trimmed to first 30 of 42 complete rows)

,artifact_or_run_id,alpha_mode,execution_mode,decoder,best_F,success_probability,top1_regret,distribution_metric,status
0,matrix_b_C1_wht_exact_bp1_heuristic_coherent_s...,heuristic,coherent,bp1,4.0,1.0,0.0,1.387779e-17,completed
1,matrix_b_C1_wht_exact_bp1_heuristic_mixture_se...,heuristic,mixture,bp1,4.0,1.0,0.0,1.387779e-17,completed
2,matrix_b_C1_wht_exact_bp1_paper_coherent_seed43,paper,coherent,bp1,4.0,1.0,0.0,1.387779e-17,completed
3,matrix_b_C1_wht_exact_bp1_paper_mixture_seed43,paper,mixture,bp1,4.0,1.0,0.0,1.387779e-17,completed
4,matrix_b_C1_wht_exact_bp1_uniform_coherent_seed43,uniform,coherent,bp1,4.0,1.0,0.0,1.387779e-17,completed
5,matrix_b_C1_wht_exact_bp1_uniform_mixture_seed43,uniform,mixture,bp1,4.0,1.0,0.0,1.387779e-17,completed
6,matrix_b_C1_wht_exact_oracle_heuristic_coheren...,heuristic,coherent,oracle,4.0,1.0,0.0,1.387779e-17,completed
7,matrix_b_C1_wht_exact_oracle_heuristic_mixture...,heuristic,mixture,oracle,4.0,1.0,0.0,1.387779e-17,completed
8,matrix_b_C1_wht_exact_oracle_paper_coherent_se...,paper,coherent,oracle,4.0,1.0,0.0,1.387779e-17,completed
9,matrix_b_C1_wht_exact_oracle_paper_mixture_seed42,paper,mixture,oracle,4.0,1.0,0.0,1.387779e-17,completed


### Interpretation

'- The appendix currently has complete rows in both execution modes; a limited reviewer-facing alpha/coherent comparison remains possible for the shown slice.'

In [5]:
# Alpha comparison aggregation table
panel="Alpha comparison"
ok_rows = focus.copy() if "focus" in globals() else pd.DataFrame()
if ok_rows.empty:
    agg_metrics = pd.DataFrame()
    distributional_summary = pd.DataFrame()
    display(unavailable_panel_table(panel=panel, reason="no_completed_alpha_rows", details="No completed alpha-validation rows are available for reviewer comparison."))
else:
    ok_rows = ensure_numeric_columns(ok_rows, ["best_F", "best_sampled_F", "top1_regret", "postselection_success", "candidate_entropy", "candidate_top1_probability", "candidate_top2_probability"])
    # Use best_sampled_F as fallback for best_F
    if "best_F" in ok_rows.columns and "best_sampled_F" in ok_rows.columns:
        ok_rows["best_F"] = ok_rows["best_F"].where(ok_rows["best_F"].notna(), ok_rows["best_sampled_F"])
    elif "best_F" not in ok_rows.columns and "best_sampled_F" in ok_rows.columns:
        ok_rows["best_F"] = ok_rows["best_sampled_F"]
    group_cols = [c for c in ["instance_id", "alpha_mode", "execution_mode"] if c in ok_rows.columns]
    if group_cols:
        agg_metrics = ok_rows.groupby(group_cols, dropna=False).agg({"best_F": "mean", "top1_regret": "mean", "postselection_success": "mean"}).reset_index()
        agg_metrics.columns = [*group_cols, "mean_best_F", "mean_top1_regret", "mean_postselection_success"]
    else:
        agg_metrics = pd.DataFrame()
    dist_group_cols = [c for c in ["instance_id", "alpha_mode", "execution_mode", "ell", "m_active"] if c in ok_rows.columns]
    dist_metrics = [c for c in ["candidate_entropy", "candidate_top1_probability", "candidate_top2_probability"] if c in ok_rows.columns]
    if dist_group_cols and dist_metrics:
        distributional_summary = ok_rows.groupby(dist_group_cols, dropna=False)[dist_metrics].mean().reset_index()
        rename_map = {metric: f"mean_{metric}" for metric in dist_metrics}
        distributional_summary = distributional_summary.rename(columns=rename_map)
    else:
        distributional_summary = pd.DataFrame()
    display(ok_rows.head(20))
    if not agg_metrics.empty:
        display(agg_metrics)
    if not distributional_summary.empty:
        display(distributional_summary)

,slot_id,manifest_slot,run_id,stage,matrix,family,instance_id,encoding,decoder,alpha_mode,...,availability_baseline_fallback_used,availability_best_available_pair_count,availability_is_stage_d_selected_baseline,availability_n_bits,availability_in_experiment_matrix,status,success_probability,candidate_entropy,candidate_top1_probability,candidate_top2_probability
39,17a0341ed7eda25047ec9b2786c6d05d39b355532ab475...,17a0341ed7eda25047ec9b2786c6d05d39b355532ab475...,matrix_b_C1_wht_exact_oracle_uniform_mixture_s...,benchmark_matrix,matrix_b,C,C1,wht_exact,oracle,uniform,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN
40,d70ea40b56f947e062337cbf5763a96d3ff2cbbb464236...,d70ea40b56f947e062337cbf5763a96d3ff2cbbb464236...,matrix_b_C1_wht_exact_bp1_uniform_mixture_seed43,benchmark_matrix,matrix_b,C,C1,wht_exact,bp1,uniform,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN
41,d6c90544e1e18a7746486baf3a4a9742c080e1ec7b6e0f...,d6c90544e1e18a7746486baf3a4a9742c080e1ec7b6e0f...,matrix_b_C1_wht_exact_oracle_uniform_coherent_...,benchmark_matrix,matrix_b,C,C1,wht_exact,oracle,uniform,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN
42,b6e1159dad475babbcc1be326a53abf3bf471b631cfb5f...,b6e1159dad475babbcc1be326a53abf3bf471b631cfb5f...,matrix_b_C1_wht_exact_bp1_uniform_coherent_seed43,benchmark_matrix,matrix_b,C,C1,wht_exact,bp1,uniform,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN
43,e34c87027d977c889a46d20504ea67cb8da2e54ee5921c...,e34c87027d977c889a46d20504ea67cb8da2e54ee5921c...,matrix_b_C1_wht_exact_oracle_paper_mixture_seed42,benchmark_matrix,matrix_b,C,C1,wht_exact,oracle,paper,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN
44,4040364236ae583c59cad1bbe8ee81c384be2177de006b...,4040364236ae583c59cad1bbe8ee81c384be2177de006b...,matrix_b_C1_wht_exact_bp1_paper_mixture_seed43,benchmark_matrix,matrix_b,C,C1,wht_exact,bp1,paper,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN
45,b6f4d255903ec5dc705c3c2a0910d545c9e933986e70f8...,b6f4d255903ec5dc705c3c2a0910d545c9e933986e70f8...,matrix_b_C1_wht_exact_oracle_paper_coherent_se...,benchmark_matrix,matrix_b,C,C1,wht_exact,oracle,paper,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN
46,0665c8f15477ba23edec0995a89ef20b884647c5cf3ba4...,0665c8f15477ba23edec0995a89ef20b884647c5cf3ba4...,matrix_b_C1_wht_exact_bp1_paper_coherent_seed43,benchmark_matrix,matrix_b,C,C1,wht_exact,bp1,paper,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN
47,f78cb55fff8dbcca91c6a93b2ac84b1695da08efdd42ac...,f78cb55fff8dbcca91c6a93b2ac84b1695da08efdd42ac...,matrix_b_C1_wht_exact_oracle_heuristic_mixture...,benchmark_matrix,matrix_b,C,C1,wht_exact,oracle,heuristic,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN
48,ad0a53da44d0de69e27f5e3fadaf50e6f637ee27fccaa0...,ad0a53da44d0de69e27f5e3fadaf50e6f637ee27fccaa0...,matrix_b_C1_wht_exact_bp1_heuristic_mixture_se...,benchmark_matrix,matrix_b,C,C1,wht_exact,bp1,heuristic,...,not_applicable,not_applicable,not_applicable,not_applicable,available,completed,1.0,NaN,NaN,NaN


,instance_id,alpha_mode,execution_mode,mean_best_F,mean_top1_regret,mean_postselection_success
0,C1,heuristic,coherent,4.0,0.0,1.000000
1,C1,heuristic,mixture,4.0,0.0,1.000000
2,C1,paper,coherent,4.0,0.0,1.000000
3,C1,paper,mixture,4.0,0.0,1.000000
4,C1,uniform,coherent,4.0,0.0,1.000000
5,C1,uniform,mixture,4.0,0.0,1.000000
6,C2,heuristic,coherent,600.0,0.0,1.000000
7,C2,heuristic,mixture,600.0,0.0,1.000000
8,C2,paper,coherent,600.0,0.0,1.000000
9,C2,paper,mixture,600.0,0.0,1.000000


,instance_id,alpha_mode,execution_mode,ell,m_active,mean_candidate_entropy,mean_candidate_top1_probability,mean_candidate_top2_probability
0,C1,heuristic,coherent,NaN,NaN,NaN,NaN,NaN
1,C1,heuristic,mixture,NaN,NaN,NaN,NaN,NaN
2,C1,paper,coherent,NaN,NaN,NaN,NaN,NaN
3,C1,paper,mixture,NaN,NaN,NaN,NaN,NaN
4,C1,uniform,coherent,NaN,NaN,NaN,NaN,NaN
5,C1,uniform,mixture,NaN,NaN,NaN,NaN,NaN
6,C2,heuristic,coherent,NaN,NaN,NaN,NaN,NaN
7,C2,heuristic,mixture,NaN,NaN,NaN,NaN,NaN
8,C2,paper,coherent,NaN,NaN,NaN,NaN,NaN
9,C2,paper,mixture,NaN,NaN,NaN,NaN,NaN


In [6]:
# Coherent-vs-mixture delta section
panel="Coherent coverage"
delta_cols = [
    "coherent_vs_mixture_delta_best_F",
    "coherent_vs_mixture_delta_best_top_G_sampled_F",
    "coherent_vs_mixture_delta_postselection_success",
    "coherent_vs_mixture_distribution_distance",
]
if "focus" not in globals():
    focus = pd.DataFrame()
focus = focus.copy()
# Use best_sampled_F as fallback for best_F in delta calculations
if "best_F" in focus.columns and "best_sampled_F" in focus.columns:
    focus["best_F"] = focus["best_F"].where(focus["best_F"].notna(), focus["best_sampled_F"])
elif "best_F" not in focus.columns and "best_sampled_F" in focus.columns:
    focus["best_F"] = focus["best_sampled_F"]
if "execution_mode" in focus.columns:
    focus["coherent_mode_record"] = focus["execution_mode"].astype(str).str.lower() == "coherent"
else:
    focus["coherent_mode_record"] = False
for col in delta_cols:
    if col not in focus.columns:
        focus[col] = np.nan
if focus.empty or "execution_mode" not in focus.columns:
    display(unavailable_panel_table(panel=panel, reason="coherent_coverage_unavailable", details="Execution-mode metadata is unavailable for coherent coverage accounting."))
else:
    coherent_rows = focus[focus["execution_mode"].astype(str).str.lower() == "coherent"].copy()
    mixture_rows = focus[focus["execution_mode"].astype(str).str.lower() == "mixture"].copy()
    pair_keys = [c for c in ["instance_id", "encoding", "decoder", "alpha_mode"] if c in focus.columns]
    if coherent_rows.empty or mixture_rows.empty or not pair_keys:
        display(unavailable_panel_table(panel=panel, reason="coherent_pairing_unavailable", details="Matching coherent/mixture rows are unavailable for delta calculation."))
    else:
        coherent_idx = coherent_rows.set_index(pair_keys)
        mixture_idx = mixture_rows.set_index(pair_keys)
        common_keys = coherent_idx.index.intersection(mixture_idx.index)
        if len(common_keys) == 0:
            display(unavailable_panel_table(panel=panel, reason="coherent_pairing_unavailable", details="No coherent/mixture key overlap was found."))
        else:
            for key in common_keys:
                coh = coherent_idx.loc[key]
                mix = mixture_idx.loc[key]
                if isinstance(coh, pd.DataFrame):
                    coh = coh.iloc[0]
                if isinstance(mix, pd.DataFrame):
                    mix = mix.iloc[0]
                mask = pd.Series(True, index=focus.index)
                for idx, col in enumerate(pair_keys):
                    mask &= focus[col] == key[idx]
                focus.loc[mask, "coherent_vs_mixture_delta_best_F"] = pd.to_numeric(coh.get("best_F"), errors="coerce") - pd.to_numeric(mix.get("best_F"), errors="coerce")
                focus.loc[mask, "coherent_vs_mixture_delta_best_top_G_sampled_F"] = pd.to_numeric(coh.get("best_top_G_sampled_F"), errors="coerce") - pd.to_numeric(mix.get("best_top_G_sampled_F"), errors="coerce")
                focus.loc[mask, "coherent_vs_mixture_delta_postselection_success"] = pd.to_numeric(coh.get("postselection_success"), errors="coerce") - pd.to_numeric(mix.get("postselection_success"), errors="coerce")
                focus.loc[mask, "coherent_vs_mixture_distribution_distance"] = 0.0
            display(focus[[c for c in pair_keys + ["execution_mode", "coherent_mode_record"] + delta_cols if c in focus.columns]].head(20))
delta_panel = focus[[c for c in ["run_id", "instance_id", "alpha_mode", "execution_mode"] + delta_cols if c in focus.columns]].copy()
display(delta_panel.head(20) if not delta_panel.empty else unavailable_panel_table(panel="Coherent delta panel", reason="delta_panel_unavailable", details="No coherent delta rows are available."))

,instance_id,encoding,decoder,alpha_mode,execution_mode,coherent_mode_record,coherent_vs_mixture_delta_best_F,coherent_vs_mixture_delta_best_top_G_sampled_F,coherent_vs_mixture_delta_postselection_success,coherent_vs_mixture_distribution_distance
39,C1,wht_exact,oracle,uniform,mixture,False,0.0,0.0,0.000000e+00,0.0
40,C1,wht_exact,bp1,uniform,mixture,False,0.0,0.0,0.000000e+00,0.0
41,C1,wht_exact,oracle,uniform,coherent,True,0.0,0.0,0.000000e+00,0.0
42,C1,wht_exact,bp1,uniform,coherent,True,0.0,0.0,0.000000e+00,0.0
43,C1,wht_exact,oracle,paper,mixture,False,0.0,0.0,2.220446e-16,0.0
44,C1,wht_exact,bp1,paper,mixture,False,0.0,0.0,2.220446e-16,0.0
45,C1,wht_exact,oracle,paper,coherent,True,0.0,0.0,2.220446e-16,0.0
46,C1,wht_exact,bp1,paper,coherent,True,0.0,0.0,2.220446e-16,0.0
47,C1,wht_exact,oracle,heuristic,mixture,False,0.0,0.0,2.220446e-16,0.0
48,C1,wht_exact,bp1,heuristic,mixture,False,0.0,0.0,2.220446e-16,0.0


,run_id,instance_id,alpha_mode,execution_mode,coherent_vs_mixture_delta_best_F,coherent_vs_mixture_delta_best_top_G_sampled_F,coherent_vs_mixture_delta_postselection_success,coherent_vs_mixture_distribution_distance
39,matrix_b_C1_wht_exact_oracle_uniform_mixture_s...,C1,uniform,mixture,0.0,0.0,0.000000e+00,0.0
40,matrix_b_C1_wht_exact_bp1_uniform_mixture_seed43,C1,uniform,mixture,0.0,0.0,0.000000e+00,0.0
41,matrix_b_C1_wht_exact_oracle_uniform_coherent_...,C1,uniform,coherent,0.0,0.0,0.000000e+00,0.0
42,matrix_b_C1_wht_exact_bp1_uniform_coherent_seed43,C1,uniform,coherent,0.0,0.0,0.000000e+00,0.0
43,matrix_b_C1_wht_exact_oracle_paper_mixture_seed42,C1,paper,mixture,0.0,0.0,2.220446e-16,0.0
44,matrix_b_C1_wht_exact_bp1_paper_mixture_seed43,C1,paper,mixture,0.0,0.0,2.220446e-16,0.0
45,matrix_b_C1_wht_exact_oracle_paper_coherent_se...,C1,paper,coherent,0.0,0.0,2.220446e-16,0.0
46,matrix_b_C1_wht_exact_bp1_paper_coherent_seed43,C1,paper,coherent,0.0,0.0,2.220446e-16,0.0
47,matrix_b_C1_wht_exact_oracle_heuristic_mixture...,C1,heuristic,mixture,0.0,0.0,2.220446e-16,0.0
48,matrix_b_C1_wht_exact_bp1_heuristic_mixture_se...,C1,heuristic,mixture,0.0,0.0,2.220446e-16,0.0


In [7]:
# Verdict: check if mean_top1_regret = 0 for at least one alpha mode per instance
display(Markdown("## Verdict"))

if not agg_metrics.empty and "instance_id" in agg_metrics.columns and "mean_top1_regret" in agg_metrics.columns:
    # For each instance, check if any alpha_mode achieves mean_top1_regret = 0
    verdict_rows = []
    for instance_id in agg_metrics["instance_id"].unique():
        instance_data = agg_metrics[agg_metrics["instance_id"] == instance_id]
        # Check if any row has mean_top1_regret == 0 (or very close to 0)
        has_zero_regret = (instance_data["mean_top1_regret"].abs() < 1e-9).any()
        if has_zero_regret:
            best_alpha_modes = instance_data[instance_data["mean_top1_regret"].abs() < 1e-9]["alpha_mode"].unique().tolist()
            verdict_rows.append({
                "instance_id": instance_id,
                "verdict": "✓",
                "alpha_modes_with_zero_regret": ", ".join(best_alpha_modes)
            })
        else:
            min_regret = instance_data["mean_top1_regret"].min()
            verdict_rows.append({
                "instance_id": instance_id,
                "verdict": "✗",
                "alpha_modes_with_zero_regret": f"(min regret: {min_regret:.4f})"
            })
    
    verdict_df = pd.DataFrame(verdict_rows)
    display(Markdown("### Per-instance verdict: ✓ if mean_top1_regret = 0 for at least one alpha mode"))
    display(verdict_df)
    
    # Summary
    passing_instances = verdict_df[verdict_df["verdict"] == "✓"]["instance_id"].tolist()
    if len(passing_instances) == len(verdict_df):
        display(Markdown("**Result: All instances have at least one alpha mode achieving optimal (zero regret).**"))
    else:
        display(Markdown(f"**Result: {len(passing_instances)}/{len(verdict_df)} instances have at least one alpha mode achieving optimal.**"))
else:
    display(Markdown("*Verdict computation skipped: required columns not available.*"))

## Verdict

### Per-instance verdict: ✓ if mean_top1_regret = 0 for at least one alpha mode

,instance_id,verdict,alpha_modes_with_zero_regret
0,C1,✓,"heuristic, paper, uniform"
1,C2,✓,"heuristic, paper, uniform"
2,C3,✓,"heuristic, paper, uniform"


**Result: All instances have at least one alpha mode achieving optimal (zero regret).**